In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from os.path import getsize

In [ ]:
df_train = pd.read_parquet("03_datasets/mean_train.parquet")
df_test = pd.read_parquet("03_datasets/mean_test.parquet")

df_train["FirstMoves"] = df_train["FirstMoves"].str.split(" ")
df_test["FirstMoves"] = df_test["FirstMoves"].str.split(" ")

elos = df_train["Elo"].values
moves = df_train["FirstMoves"].values

In [ ]:
line_count = {}
line_rating = {}

for elo, move in zip(elos, moves):
    
    for i in range(1, min(20, len(move))):
        
        key = " ".join(move[:i])
        line_count[key] = line_count.get(key, 0) + 1
        line_rating[key] = line_rating.get(key, 0) + elo

In [ ]:
result = {
    k: v / line_count[k]
    for k, v in line_rating.items()
    if line_count[k] > 100
}

In [ ]:
series = pd.Series(result).sort_values()

In [ ]:
# px.line(series.values, template="plotly_white")

In [ ]:
# series.sort_index().to_dict()

In [ ]:
tree = {}
for key, value in series.sort_index().to_dict().items():
    current_node = tree
    for move in key.split(" "):
        if not (move in current_node):
            current_node[move] = {}
        current_node = current_node[move]
    current_node["mean"] = value

In [ ]:
tree

In [ ]:
tree["e4"]["e5"]["Bc4"]["mean"]

In [ ]:
def get_line_mean(moves_list):
    try:
        current_node = tree
        for move in moves_list:
            if len(current_node) == 1:
                return current_node["mean"]
            if move in current_node:
                current_node = current_node[move]
            else:
                return current_node["mean"]
    except:
        return 1500

In [ ]:
df_train["LineTreeMean"] = df_train["FirstMoves"].map(get_line_mean).fillna(1500)
df_test["LineTreeMean"] = df_test["FirstMoves"].map(get_line_mean).fillna(1500)

In [ ]:
# px.histogram(df_train["LineTreeMean"], template="plotly_white")

In [ ]:
df_train.drop(columns=["FirstMoves"]).to_parquet("03_datasets/final_train.parquet")
df_test.drop(columns=["FirstMoves"]).to_parquet("03_datasets/final_test.parquet")